In [1]:
"""
EU AI Act — Best-Practice Chunking Pipeline  v2
================================================

Produces three types of chunks, each with rich typed metadata:

  TYPE 1 — RECITAL   : one chunk per recital (1-180), pages 1-44
  TYPE 2 — ARTICLE   : one chunk per article (1-113), pages 44-123
                       large articles are split with RecursiveCharacterTextSplitter
                       every sub-chunk inherits the full parent metadata
  TYPE 3 — ANNEX     : one chunk per annex (I-XIII), pages 124-144

Each chunk is a dict:
  {
    "page_content": "...",
    "metadata": {
      "content_type": "recital" | "article" | "annex",
      "page_num":     "12",
      "source":       "human-readable citation string",
      "<content_type>": { ...type-specific nested metadata... }
    }
  }

The one-hop cross-reference field
----------------------------------
Every chunk carries "referenced_articles" and "referenced_annexes".
Use these at query time to do a one-hop fetch:
  1. retrieve the primary chunk(s) that answer the query
  2. fetch all chunks whose article_num appears in referenced_articles
  3. pass the combined context to the LLM

Optional LLM step  (--summarise flag)
--------------------------------------
Adds per-article, per-chapter, per-section, per-annex summaries stored
in the metadata AND in hierarchy.json.  These power future agents that
decide which articles to retrieve without running a full vector search.

Outputs (written to --output directory)
----------------------------------------
  chunks_recitals.json
  chunks_articles.json
  chunks_annexes.json
  hierarchy.json        chapter > section > article map (+ optional summaries)
  all_chunks.json       everything merged, ready for vector store

Usage
-----
  python eu_ai_act_chunker.py --pdf eu_ai_act.pdf
  python eu_ai_act_chunker.py --pdf eu_ai_act.pdf --summarise
"""

import re
import json
import copy
import argparse
from pathlib import Path

import pdfplumber
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter


# ─────────────────────────────────────────────────────────────────────────────
# LOOKUP TABLES  (verified against the PDF)
# ─────────────────────────────────────────────────────────────────────────────

# The PDF renders chapter headings without spaces ("PROHIBITEDAIPRACTICES").
# We use a lookup table keyed on the roman numeral to get the proper title.
CHAPTER_TITLES = {
    'I':    'GENERAL PROVISIONS',
    'II':   'PROHIBITED AI PRACTICES',
    'III':  'HIGH-RISK AI SYSTEMS',
    'IV':   'TRANSPARENCY OBLIGATIONS FOR PROVIDERS AND DEPLOYERS OF CERTAIN AI SYSTEMS',
    'V':    'GENERAL-PURPOSE AI MODELS',
    'VI':   'MEASURES IN SUPPORT OF INNOVATION',
    'VII':  'GOVERNANCE',
    'VIII': 'EU DATABASE FOR HIGH-RISK AI SYSTEMS',
    'IX':   'POST-MARKET MONITORING, INFORMATION SHARING AND MARKET SURVEILLANCE',
    'X':    'CODES OF CONDUCT AND GUIDELINES',
    'XI':   'DELEGATION OF POWER AND COMMITTEE PROCEDURE',
    'XII':  'PENALTIES',
    'XIII': 'FINAL PROVISIONS',
}

ANNEX_PURPOSE = {
    'I':    'List of Union harmonisation legislation that triggers high-risk classification (Art. 6)',
    'II':   'List of criminal offences enabling real-time remote biometric ID (Art. 5)',
    'III':  'Master list of high-risk AI use-case areas (Art. 6(2)) — biometrics, critical infrastructure, education, employment, law enforcement, migration, administration of justice',
    'IV':   'Required contents of technical documentation for high-risk AI systems (Art. 11)',
    'V':    'Template for EU declaration of conformity (Art. 47)',
    'VI':   'Conformity assessment procedure — internal control (Art. 43)',
    'VII':  'Conformity assessment procedure — quality management system + technical documentation (Art. 43)',
    'VIII': 'Registration information for high-risk AI systems (Art. 49)',
    'IX':   'Registration info for Annex III systems tested in real-world conditions (Art. 60)',
    'X':    'Large-scale IT systems in the area of Freedom, Security and Justice',
    'XI':   'Technical documentation for general-purpose AI model providers to submit to AI Office (Art. 53)',
    'XII':  'Transparency information for general-purpose AI model providers to downstream providers (Art. 53)',
    'XIII': 'Criteria for designating general-purpose AI models with systemic risk (Art. 51)',
}

ANNEX_ORDER = ['I','II','III','IV','V','VI','VII','VIII','IX','X','XI','XII','XIII']


# ─────────────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def _clean(text: str) -> str:
    """Strip PDF running headers and footers from page text."""
    text = re.sub(r'OJ L,\s*\d+\.\d+\.\d+\s*\n', '', text)
    text = re.sub(r'ELI:\s*http://[^\n]+\n', '', text)
    text = re.sub(r'\d+/144\s*\n', '', text)
    text = re.sub(r'\nEN\s*\n', '\n', text)
    return text


def _build_stream(pdf, page_range: range) -> tuple:
    """
    Concatenate cleaned text from PDF pages.
    Returns (full_text, page_map) where page_map = [(char_offset, oj_page), ...].
    """
    full_text = ""
    page_map = []
    for pg_idx in page_range:
        page = pdf.pages[pg_idx]
        raw = page.extract_text() or ""
        oj_match = re.search(r'(\d+)/144', raw)
        oj_page = int(oj_match.group(1)) if oj_match else pg_idx + 1
        cleaned = _clean(raw)
        page_map.append((len(full_text), oj_page))
        full_text += cleaned
    return full_text, page_map


def _page_at(offset: int, page_map: list) -> int:
    """Return OJ page number containing character offset."""
    result = page_map[0][1]
    for char_start, oj_page in page_map:
        if char_start <= offset:
            result = oj_page
        else:
            break
    return result


def _article_refs(text: str) -> list:
    return sorted(set(re.findall(r'Article \d+', text)))


def _annex_refs(text: str) -> list:
    return sorted(set(re.findall(r'Annex [IVX]+', text)))


def _roman_to_int(r: str) -> int:
    vals = {'I': 1, 'V': 5, 'X': 10, 'L': 50, 'C': 100}
    result, prev = 0, 0
    for ch in reversed(r.upper().strip()):
        v = vals.get(ch, 0)
        result += v if v >= prev else -v
        prev = v
    return result


# ─────────────────────────────────────────────────────────────────────────────
# STEP 1 — RECITALS  (OJ pages 1-44, PDF indices 0-43)
# ─────────────────────────────────────────────────────────────────────────────

def extract_recitals(pdf) -> list:
    """
    Parse all 180 recitals, one chunk each.

    Detection: recital markers look like  \\n(N) CapitalLetter
    Footnote refs have the same pattern, so we walk candidates and
    accept only the strictly sequential ones: 1, 2, 3, ..., 180.

    Metadata shape
    --------------
    content_type : "recital"
    page_num     : "12"
    source       : "Recital (43) — EU AI Act (p. 12)"
    recital      : {
        recital_num         : 43
        page_num            : 12
        referenced_articles : ["Article 5"]
        referenced_annexes  : []
        summary             : ""   <- LLM fills this if --summarise
    }
    """
    full_text, page_map = _build_stream(pdf, range(0, 44))

    pattern = re.compile(r'\n\((\d+)\)\s+[A-Z]')
    candidates = [
        (int(m.group(1)), m.start(), m.start() + len(m.group(0)) - 1)
        for m in pattern.finditer(full_text)
    ]

    real = []
    expected = 1
    for num, marker_start, text_start in candidates:
        if num == expected:
            real.append((num, marker_start, text_start))
            expected += 1

    if len(real) != 180:
        print(f"[WARN] Expected 180 recitals, found {len(real)}")

    chunks = []
    for i, (num, marker_start, text_start) in enumerate(real):
        end  = real[i + 1][1] if i + 1 < len(real) else len(full_text)
        body = full_text[text_start:end].strip()
        page = _page_at(marker_start, page_map)

        chunks.append({
            'page_content': body,
            'metadata': {
                'content_type': 'recital',
                'page_num':     str(page),
                'source':       f'Recital ({num}) — EU AI Act (p. {page})',
                'recital': {
                    'recital_num':         num,
                    'page_num':            page,
                    'referenced_articles': _article_refs(body),
                    'referenced_annexes':  _annex_refs(body),
                    'summary':             '',
                },
            },
        })

    return chunks


# ─────────────────────────────────────────────────────────────────────────────
# STEP 2 — ARTICLES  (OJ pages 44-123, PDF indices 43-123)
# ─────────────────────────────────────────────────────────────────────────────

def _build_article_context(full_text: str, page_map: list) -> list:
    """
    Walk the operative text once, tracking the current chapter/section context.
    Returns a list of article dicts (sorted by article_num) with positional info.
    """
    # Chapter heading: "\nCHAPTERII\n" (no space in PDF) or "\nCHAPTER II\n"
    ch_pattern  = re.compile(r'\n(CHAPTER\s*([IVX]+))\n')
    sec_pattern = re.compile(r'\n(SECTION\s*(\d+))\n([^\n]+)')
    art_pattern = re.compile(r'\nArticle (\d+)\n(.+?)\n')

    events = []
    for m in ch_pattern.finditer(full_text):
        events.append((m.start(), 'chapter', (m.group(2),)))
    for m in sec_pattern.finditer(full_text):
        events.append((m.start(), 'section', (m.group(2), m.group(3).strip())))
    for m in art_pattern.finditer(full_text):
        events.append((m.start(), 'article',
                        (int(m.group(1)), m.group(2).strip(), m.end())))
    events.sort(key=lambda e: e[0])

    cur_roman     = ''
    cur_sec_num   = ''
    cur_sec_title = ''
    ctx_list      = []

    for offset, kind, payload in events:
        if kind == 'chapter':
            cur_roman     = payload[0]
            cur_sec_num   = ''
            cur_sec_title = ''
        elif kind == 'section':
            cur_sec_num, cur_sec_title = payload
        elif kind == 'article':
            art_num, art_title, body_start = payload
            ctx_list.append({
                'article_num':   art_num,
                'article_title': art_title,
                'chapter_num':   f'CHAPTER {cur_roman}' if cur_roman else '',
                'chapter_title': CHAPTER_TITLES.get(cur_roman, cur_roman),
                'section_num':   f'SECTION {cur_sec_num}' if cur_sec_num else '',
                'section_title': cur_sec_title,
                'page_num':      _page_at(offset, page_map),
                '_marker_start': offset,
                '_body_start':   body_start,
            })

    ctx_list.sort(key=lambda a: a['article_num'])
    return ctx_list


def extract_articles(pdf) -> list:
    """
    Parse all articles (one chunk per article, before splitting).

    Metadata shape
    --------------
    content_type : "article"
    page_num     : "51"
    source       : "Article 5 — Prohibited AI practices | CHAPTER II (p. 51)"
    article      : {
        article_num         : 5
        article_title       : "Prohibited AI practices"
        chapter_num         : "CHAPTER II"
        chapter_title       : "PROHIBITED AI PRACTICES"
        section_num         : ""
        section_title       : ""
        page_num            : 51
        referenced_articles : ["Article 27", "Article 9"]
        referenced_annexes  : ["Annex II"]
        chunk_index         : 0    <- updated by split_large_articles()
        total_chunks        : 1    <- updated by split_large_articles()
        summary             : ""   <- LLM fills if --summarise
    }
    """
    full_text, page_map = _build_stream(pdf, range(43, 124))
    ctx_list = _build_article_context(full_text, page_map)

    chunks = []
    for i, ctx in enumerate(ctx_list):
        next_marker = (
            ctx_list[i + 1]['_marker_start']
            if i + 1 < len(ctx_list)
            else len(full_text)
        )
        body = full_text[ctx['_body_start']:next_marker].strip()

        sec_part = f' | {ctx["section_num"]}' if ctx['section_num'] else ''
        source   = (
            f"Article {ctx['article_num']} — {ctx['article_title']}"
            f" | {ctx['chapter_num']}{sec_part} (p. {ctx['page_num']})"
        )

        chunks.append({
            'page_content': body,
            'metadata': {
                'content_type': 'article',
                'page_num':     str(ctx['page_num']),
                'source':       source,
                'article': {
                    'article_num':         ctx['article_num'],
                    'article_title':       ctx['article_title'],
                    'chapter_num':         ctx['chapter_num'],
                    'chapter_title':       ctx['chapter_title'],
                    'section_num':         ctx['section_num'],
                    'section_title':       ctx['section_title'],
                    'page_num':            ctx['page_num'],
                    'referenced_articles': _article_refs(body),
                    'referenced_annexes':  _annex_refs(body),
                    'chunk_index':         0,
                    'total_chunks':        1,
                    'summary':             '',
                },
            },
        })

    return chunks


def split_large_articles(article_chunks: list, chunk_size: int = 1500,
                          chunk_overlap: int = 150) -> list:
    """
    Split articles that exceed chunk_size.
    Each sub-chunk inherits full parent metadata.
    chunk_size=1500 chars ≈ 375 tokens — fits comfortably in retrieval context.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )

    result = []
    for c in article_chunks:
        if len(c['page_content']) <= chunk_size:
            result.append(c)
            continue
        parts = splitter.split_text(c['page_content'])
        for idx, part in enumerate(parts):
            nc = copy.deepcopy(c)
            nc['page_content'] = part
            nc['metadata']['article']['chunk_index']  = idx
            nc['metadata']['article']['total_chunks'] = len(parts)
            nc['metadata']['source'] += f' [{idx+1}/{len(parts)}]'
            result.append(nc)

    return result


# ─────────────────────────────────────────────────────────────────────────────
# STEP 3 — ANNEXES  (OJ pages 124-144, PDF indices 123-143)
# ─────────────────────────────────────────────────────────────────────────────

def extract_annexes(pdf) -> list:
    """
    Parse all 13 annexes, one chunk each.

    The PDF renders "ANNEXIII" without a space. We use a longest-match roman
    numeral regex to avoid "ANNEX I" matching inside "ANNEX III" etc.

    Metadata shape
    --------------
    content_type : "annex"
    page_num     : "127"
    source       : "Annex III — High-risk AI systems referred to in Article 6(2) (p. 127)"
    annex        : {
        annex_num           : "III"
        annex_num_int       : 3
        annex_title         : "High-risk AI systems referred to in Article 6(2)"
        annex_purpose       : "Master list of high-risk AI use-case areas ..."
        page_num            : 127
        referenced_articles : ["Article 6"]
        summary             : ""   <- LLM fills if --summarise
    }
    """
    full_text, page_map = _build_stream(pdf, range(123, 144))

    roman_re = r'(?:XIII|XII|XI|IX|VIII|VII|VI|IV|III|II|I|X)'
    pattern  = re.compile(r'\nANNEX\s*(' + roman_re + r')\n([^\n]+)')

    seen = set()
    boundaries = []
    for m in pattern.finditer(full_text):
        roman = m.group(1)
        if roman not in seen:
            seen.add(roman)
            boundaries.append((roman, m.group(2).strip(), m.start(), m.end()))

    order = {r: i for i, r in enumerate(ANNEX_ORDER)}
    boundaries.sort(key=lambda b: order.get(b[0], 99))

    chunks = []
    for i, (roman, title, marker_start, body_start) in enumerate(boundaries):
        end  = boundaries[i + 1][2] if i + 1 < len(boundaries) else len(full_text)
        body = full_text[body_start:end].strip()
        page = _page_at(marker_start, page_map)

        chunks.append({
            'page_content': body,
            'metadata': {
                'content_type': 'annex',
                'page_num':     str(page),
                'source':       f'Annex {roman} — {title} (p. {page})',
                'annex': {
                    'annex_num':           roman,
                    'annex_num_int':       _roman_to_int(roman),
                    'annex_title':         title,
                    'annex_purpose':       ANNEX_PURPOSE.get(roman, ''),
                    'page_num':            page,
                    'referenced_articles': _article_refs(body),
                    'summary':             '',
                },
            },
        })

    return chunks


# ─────────────────────────────────────────────────────────────────────────────
# STEP 4 — HIERARCHY INDEX
# ─────────────────────────────────────────────────────────────────────────────

def build_hierarchy(article_chunks: list) -> dict:
    """
    Build: chapter > section > [articles] map.

    Agents use this to navigate without full vector search:
      "user asks about high-risk systems" → agent looks up Chapter III summary
      → decides to fetch Section 2 articles → targeted retrieval

    Shape:
    {
      "CHAPTER II": {
        "chapter_title": "PROHIBITED AI PRACTICES",
        "summary": "",
        "sections": {
          "": {
            "section_title": "",
            "summary": "",
            "articles": [
              {"article_num": 5, "article_title": "Prohibited AI practices",
               "page_num": 51, "summary": ""}
            ]
          }
        }
      }
    }
    """
    hierarchy = {}
    for c in article_chunks:
        a   = c['metadata']['article']
        ch  = a['chapter_num']
        sec = a['section_num']

        if ch not in hierarchy:
            hierarchy[ch] = {
                'chapter_title': a['chapter_title'],
                'summary': '',
                'sections': {},
            }
        if sec not in hierarchy[ch]['sections']:
            hierarchy[ch]['sections'][sec] = {
                'section_title': a['section_title'],
                'summary': '',
                'articles': [],
            }

        existing = {x['article_num'] for x in hierarchy[ch]['sections'][sec]['articles']}
        if a['article_num'] not in existing:
            hierarchy[ch]['sections'][sec]['articles'].append({
                'article_num':   a['article_num'],
                'article_title': a['article_title'],
                'page_num':      a['page_num'],
                'summary':       '',
            })

    return hierarchy


# ─────────────────────────────────────────────────────────────────────────────
# STEP 5 (OPTIONAL) — LLM SUMMARIES
# ─────────────────────────────────────────────────────────────────────────────

def _llm(system: str, user: str) -> str:
    try:
        import anthropic
        client = anthropic.Anthropic()
        resp = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=256,
            messages=[{"role": "user", "content": f"{system}\n\n{user}"}],
        )
        return resp.content[0].text.strip()
    except Exception as e:
        return f"[unavailable: {e}]"


_ART_SYS = (
    "EU AI law analyst. Write ONE sentence (<=40 words) summarising what "
    "this article requires, prohibits, or defines. Be precise. "
    "Do not begin with 'This article'."
)
_SEC_SYS = (
    "EU AI law analyst. Write ONE sentence (<=50 words) describing the "
    "collective purpose of this group of articles."
)
_CH_SYS = (
    "EU AI law analyst. Write TWO sentences (<=70 words) describing what "
    "this chapter covers and who it affects."
)
_ANN_SYS = (
    "EU AI law analyst. Write ONE sentence (<=35 words) describing the "
    "practical purpose of this annex for AI system providers."
)


def add_summaries(recital_chunks: list, article_chunks_raw: list,
                  annex_chunks: list, hierarchy: dict) -> None:
    """
    Mutate chunks in-place adding 'summary' fields.
    article_chunks_raw = one dict per article (pre-split), so one LLM call each.
    Recitals get a text preview — no LLM needed, they are already short prose.
    """
    # Recitals: text preview
    for c in recital_chunks:
        c['metadata']['recital']['summary'] = (
            c['page_content'].replace('\n', ' ')[:120].strip() + '…'
        )

    # Articles
    total = len(article_chunks_raw)
    art_summary_map = {}
    for i, c in enumerate(article_chunks_raw, 1):
        a     = c['metadata']['article']
        num   = a['article_num']
        title = a['article_title']
        print(f"  [{i:3}/{total}] Article {num}: {title[:45]}", end='\r')
        s = _llm(_ART_SYS, f"Article {num} — {title}\n\n{c['page_content'][:800]}")
        a['summary'] = s
        art_summary_map[num] = s
    print()

    # Sections + chapters: populate hierarchy summaries
    for ch_num, ch_data in hierarchy.items():
        for sec_num, sec_data in ch_data['sections'].items():
            # Mirror article summaries into hierarchy article list
            for art_info in sec_data['articles']:
                art_info['summary'] = art_summary_map.get(art_info['article_num'], '')
            if sec_num:
                lines = ', '.join(
                    f"Art {a['article_num']}: {a['article_title']}"
                    for a in sec_data['articles']
                )
                sec_data['summary'] = _llm(_SEC_SYS,
                    f"Section: {sec_data['section_title']}\nArticles: {lines}")

        all_arts = ', '.join(
            f"Art {a['article_num']}: {a['article_title']}"
            for sec in ch_data['sections'].values()
            for a in sec['articles']
        )[:600]
        ch_data['summary'] = _llm(_CH_SYS,
            f"Chapter: {ch_data['chapter_title']}\nArticles: {all_arts}")

    # Annexes
    for c in annex_chunks:
        ann = c['metadata']['annex']
        print(f"  Annex {ann['annex_num']}: {ann['annex_title'][:45]}", end='\r')
        ann['summary'] = _llm(_ANN_SYS,
            f"Annex {ann['annex_num']} — {ann['annex_title']}\n\n"
            f"{c['page_content'][:600]}")
    print("\nSummaries done.")

    return art_summary_map   # caller uses this to mirror into split chunks


# ─────────────────────────────────────────────────────────────────────────────
# CONVERT TO LANGCHAIN DOCUMENTS
# ─────────────────────────────────────────────────────────────────────────────

def to_langchain_docs(chunks: list) -> list:
    return [Document(page_content=c['page_content'], metadata=c['metadata'])
            for c in chunks]


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────

def main(pdf_path: str, summarise: bool, output_dir: str) -> None:
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)

    print(f"Opening {pdf_path} …")
    with pdfplumber.open(pdf_path) as pdf:
        print("  [1/3] Extracting recitals …")
        recital_chunks = extract_recitals(pdf)
        print(f"        → {len(recital_chunks)} recital chunks")

        print("  [2/3] Extracting articles …")
        article_chunks_raw = extract_articles(pdf)
        print(f"        → {len(article_chunks_raw)} articles (pre-split)")

        print("  [3/3] Extracting annexes …")
        annex_chunks = extract_annexes(pdf)
        print(f"        → {len(annex_chunks)} annex chunks")

    print("  Building hierarchy …")
    hierarchy = build_hierarchy(article_chunks_raw)

    art_summary_map = {}
    if summarise:
        print("  Adding LLM summaries …")
        art_summary_map = add_summaries(
            recital_chunks, article_chunks_raw, annex_chunks, hierarchy
        )

    print("  Splitting large articles …")
    article_chunks = split_large_articles(article_chunks_raw,
                                          chunk_size=1500, chunk_overlap=150)
    # Mirror summaries into split sub-chunks
    if summarise:
        for c in article_chunks:
            n = c['metadata']['article']['article_num']
            c['metadata']['article']['summary'] = art_summary_map.get(n, '')

    print(f"        → {len(article_chunks)} article chunks (post-split)")

    # ── Save ────────────────────────────────────────────────────────────────
    def _save(data, fname):
        p = out / fname
        with open(p, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        print(f"  Saved {p}  ({len(data)} items)")

    print("\nSaving …")
    _save(recital_chunks,  "chunks_recitals.json")
    _save(article_chunks,  "chunks_articles.json")
    _save(annex_chunks,    "chunks_annexes.json")
    with open(out / "hierarchy.json", 'w', encoding='utf-8') as f:
        json.dump(hierarchy, f, ensure_ascii=False, indent=2)
    print(f"  Saved {out / 'hierarchy.json'}")
    all_chunks = recital_chunks + article_chunks + annex_chunks
    _save(all_chunks, "all_chunks.json")

    # ── Stats ────────────────────────────────────────────────────────────────
    def _avg(lst):
        return sum(len(c['page_content']) for c in lst) / max(len(lst), 1)

    print("\n─── Stats ────────────────────────────────────────────")
    print(f"  Recital chunks : {len(recital_chunks):4d}   avg {_avg(recital_chunks):6.0f} chars")
    print(f"  Article chunks : {len(article_chunks):4d}   avg {_avg(article_chunks):6.0f} chars")
    print(f"  Annex chunks   : {len(annex_chunks):4d}   avg {_avg(annex_chunks):6.0f} chars")
    print(f"  TOTAL          : {len(all_chunks):4d}")
    print("──────────────────────────────────────────────────────")

    print("""
How to load
-----------
  from langchain_core.documents import Document
  import json

  with open("all_chunks.json") as f:
      raw = json.load(f)

  docs = [Document(page_content=c['page_content'], metadata=c['metadata'])
          for c in raw]

  # One-hop cross-reference fetch:
  def fetch_referenced(chunk, all_docs):
      nums = [int(r.split()[1]) for r in
              chunk.metadata.get(chunk.metadata['content_type'],{})
                            .get('referenced_articles', [])]
      return [d for d in all_docs
              if d.metadata['content_type'] == 'article'
              and d.metadata['article']['article_num'] in nums
              and d.metadata['article']['chunk_index'] == 0]
""")


    
if __name__ == "__main__":
    #     ap = argparse.ArgumentParser(description="EU AI Act chunking pipeline v2")
    #     ap.add_argument("--pdf",       default="eu_ai_act.pdf")
    #     ap.add_argument("--summarise", action="store_true",
    #                     help="Add LLM summaries (needs ANTHROPIC_API_KEY)")
    #     ap.add_argument("--output",    default="./chunker_output")
    #     args = ap.parse_args()
    pdf = "eu_ai_act.pdf"
    summarise = False
    output = "./chunker_output"
    main(pdf, summarise, output)

/Users/bala/python-envs/AI_Agents/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Opening eu_ai_act.pdf …
  [1/3] Extracting recitals …
        → 180 recital chunks
  [2/3] Extracting articles …
        → 113 articles (pre-split)
  [3/3] Extracting annexes …
        → 12 annex chunks
  Building hierarchy …
  Splitting large articles …
        → 260 article chunks (post-split)

Saving …
  Saved chunker_output/chunks_recitals.json  (180 items)
  Saved chunker_output/chunks_articles.json  (260 items)
  Saved chunker_output/chunks_annexes.json  (12 items)
  Saved chunker_output/hierarchy.json
  Saved chunker_output/all_chunks.json  (452 items)

─── Stats ────────────────────────────────────────────
  Recital chunks :  180   avg   1251 chars
  Article chunks :  260   avg   1141 chars
  Annex chunks   :   12   avg   3667 chars
  TOTAL          :  452
──────────────────────────────────────────────────────

How to load
-----------
  from langchain_core.documents import Document
  import json

  with open("all_chunks.json") as f:
      raw = json.load(f)

  docs = [Document(

In [2]:
from langchain_core.documents import Document
import json

with open("./chunker_output/all_chunks.json") as f:
      raw = json.load(f)

docs = [Document(page_content=c['page_content'], metadata=c['metadata'])
  for c in raw]


In [3]:
docs[:5]

[Document(metadata={'content_type': 'recital', 'page_num': '1', 'source': 'Recital (1) — EU AI Act (p. 1)', 'recital': {'recital_num': 1, 'page_num': 1, 'referenced_articles': [], 'referenced_annexes': [], 'summary': ''}}, page_content='ThepurposeofthisRegulationistoimprovethefunctioningoftheinternalmarketbylayingdownauniformlegal\nframework in particular for the development, the placing on the market, the putting into service and the use of\nartificialintelligencesystems(AIsystems)intheUnion,inaccordancewithUnionvalues,topromotetheuptakeof\nhumancentricandtrustworthyartificialintelligence(AI)whileensuringahighlevelofprotectionofhealth,safety,\nfundamental rights as enshrined in the Charter of Fundamental Rights of the European Union (the ‘Charter’),\nincluding democracy, the rule of law and environmental protection, to protect against the harmful effects of AI\nsystems in the Union, and to support innovation. This Regulation ensures the free movement, cross-border, of\nAI-based goods 

In [15]:


import pdfplumber

file = "./eu_ai_act.pdf"

with pdfplumber.open(file) as pdf:
    text = pdf.pages[0].extract_text
    print(text)


<bound method Page.extract_text of <Page:1>>
